# `Pattern 1` retrieval & generation - AutoAI Notebook <font size="3">ver.{version}</font>

Consider these tips for working with an auto-generated notebook:
- Notebook code generated with AutoRAG will execute successfully. If you modify the notebook, we cannot guarantee it will run successfully.
- This RAG pattern is optimized for the original data set. The pattern might fail or produce sub-optimal results if used with different data. If you want to use a different data set, consider rerunning the AutoRAG experiment to generate a new pipeline. For more information, see {platform_link}.


<a id="content"></a>
## Notebook content

This notebook contains a python code for building retrieval & generation pattern. This notebook introduces commands for retrieving chunks, building prompt and generating answers.

Some familiarity with Python is helpful. This notebook uses Python 3.13.

## Notebook goals

- Test generated RAG pattern
- Evaluate generated RAG pattern

#### About Retrieval Augmented Generation
Retrieval Augmented Generation (RAG) is a versatile pattern that can unlock a number of use cases requiring factual recall of information, such as querying a knowledge base in natural language.

In its simplest form, RAG requires 3 steps:

- Index knowledge base passages (once)
- Retrieve relevant passage(s) from knowledge base (for every user query)
- Generate a response by feeding retrieved passage into a large language model (for every user query)

This notebook covers steps 2 & 3.

## Contents

This notebook contains the following parts:

**[Setup](#setup)**
<br>&nbsp;&nbsp;[Package installation](#install)
<br>&nbsp;&nbsp;[AutoAI experiment metadata](#variables_definition)
<br>&nbsp;&nbsp;[Watsonx.ai connection](#connection)
<br>&nbsp;&nbsp;[Deployment space](#space)
<br>&nbsp;&nbsp;[Read benchmarking data](#read-benchmarking-data)

**[Inference endpoint building](#inference)**
<br>&nbsp;&nbsp;[Inference service initialization](#inference-service-initialization)
<br>&nbsp;&nbsp;[Inference service test](#customization)
<br>&nbsp;&nbsp;[Answers evaluation](#answers-evaluation)

**[Retrieval & generation test](#inference_endpoint)**
<br>&nbsp;&nbsp;[Inference service deployment](#inference-service-deployment)
<br>&nbsp;&nbsp;[Inference endpoint test](#inference-endpoint-test)<br>

**[Summary and next steps](#summary_and_next_steps)**
<br>**[Copyrights](#copyrights)**

# Setup

## Package installation
Install ai4rag package with all its dependencies.

In [1]:
!pip install "git+https://github.com/IBM/ai4rag.git@dev"

^C


## Client initialization

Instantiate LlamaStackClient for communication with llama-stack.

In [ ]:
from llama_stack_client import LlamaStackClient

client = LlamaStackClient(base_url="http://localhost:8321")

# RAG Pattern initialization

## Embedding model

Embedding model is responsible for creating vectorized form of the given chunks.
It is required to use the same embedding model that was used for adding the documents / chunks to the vector store.
If different embedding model is used, results might not be optimal.

In [ ]:
from ai4rag.rag.embedding.llama_stack import LSEmbeddingModel

embedding_model = LSEmbeddingModel(
    client=client,
    model_id="ollama/nomic-embed-text:latest",
    params={
        "context_length": 8192,
        "embedding_dimension": 768,
    }
)

## Vector store

Instance of the vector store allows to communicate with given vector database using llama-stack.
`reuse_collection_name` parameter allows to use already existing collection.

In [ ]:
from ai4rag.rag.vector_store.llama_stack import LSVectorStore

vector_store = LSVectorStore(
    client=client,
    embedding_model=embedding_model,
    provider_id="milvus",
    reuse_collection_name="vs_1234567890",
)

## Retriever

Retriever is responsible for retrieving relevant chunks from the vector store.

In [ ]:
from ai4rag.rag.retrieval.retriever import Retriever

retriever = Retriever(
    vector_store=vector_store,
    method="simple",
    number_of_chunks=5,
)

## Foundation model

Foundation model instance allows to communicate with the given LLM.
To configure foundation model with different parameters / prompt templates see the documentation.

In [ ]:
from ai4rag.search_space.src.model_props import get_system_message_text
from ai4rag.search_space.src.model_props import get_user_message_text
from ai4rag.search_space.src.model_props import get_context_template_text
from ai4rag.rag.foundation_models.foundation_model import LSFoundationModel

model_id = "ollama/llama3.2:3b"

foundation_model = LSFoundationModel(
    client=client,
    model_id=model_id,
    system_message_text=get_system_message_text(model_name=model_id),
    user_message_text=get_user_message_text(model_name=model_id),
    context_template_text=get_context_template_text(model_id)
)

## RAG Pattern

Compose instance of the RAG Pattern that can utilise all components to perform e2e retrieval augmented generation.

In [ ]:
from ai4rag.rag.template.rag_template import LlamaStackRAG

rag_pattern = LlamaStackRAG(
    foundation_model=foundation_model,
    retriever=retriever,
)

# Test RAG Pattern

# Evaluate RAG Pattern